# Phase 2 independent validation

Run this two-cell orchestrator after `phase2_implement.ipynb` or the supported publisher has created current `artifacts/phase2_outputs/phase2_summary.json` and `artifacts/phase2_operational/phase2_events.csv`. The subprocess command `python -B -m scripts.shared_monitoring_engine validate` runs `scripts.test_shared_monitoring`, rejects failed, skipped, or zero-test results, and reproduces the matched summary/events in an isolated temporary directory before comparing them with the saved outputs. It does not replay the raw Phase 1 transformation or rewrite the saved Phase 2 summary and events.

The validator, not this notebook, writes `artifacts/phase2_operational/phase2_validation.json`. It records the regression/reproduction result and source/output identities; changing a bound file can make a previously passed receipt stale. Validation first writes an `in_progress` receipt. A failed check or interruption can leave that status in place; only full success writes `passed`.

The notebook itself defines no test cases or local assertions: it launches a separate Python process and lets a nonzero exit fail the cell. The checks cover deterministic sampled-minute state, synthetic regressions, saved artifact/provenance identity, and isolated reproduction. They are not a proof of the algorithm, an inter-minute detection guarantee, a physical-network failure test, or a generalized multivariate result.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Support runs from the repository root or a subfolder by finding the
# canonical project config; fail rather than guessing if no parent has it.
ROOT = Path.cwd().resolve()
while not (ROOT / 'config' / 'project_config.yaml').is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('Run from the sensor_project repository.')
    ROOT = ROOT.parent
# Use the kernel's interpreter and suppress .pyc writes. cwd=ROOT supplies
# project-relative imports/paths; check=True surfaces a nonzero validator exit.
subprocess.run([sys.executable, '-B', '-m', 'scripts.shared_monitoring_engine', 'validate'], cwd=ROOT, check=True)